In [135]:
import sys
from math import ceil, floor

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

In [136]:
yolo_path = "./deep-learning/models/yolov3-coco/"

In [137]:
try:
    with open("./deep-learning/models/yolov3-coco/coco.names", "r") as f:
        classes = [line.strip() for line in f.readlines()]
except:
    sys.exit("You should have three required files.")

In [138]:
print(classes)

['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 'pottedplant', 'bed', 'diningtable', 'toilet', 'tvmonitor', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']


In [139]:
yolo_net = cv.dnn.readNet(yolo_path + "yolov3.weights", yolo_path + "yolov3.cfg")
layer_names = yolo_net.getLayerNames()
output_layers = [layer_names[i - 1] for i in yolo_net.getUnconnectedOutLayers()]

In [140]:
image = cv.imread("./opencv-notebook/images/05_tehran.jpg")
height, width, channels = image.shape

In [141]:
img_blob = cv.dnn.blobFromImage(image, 0.00392, (416, 416), (0, 0, 0), True, crop=False)

In [142]:
yolo_net.setInput(img_blob)
outputs = yolo_net.forward(output_layers)
outputs[0].shape

(507, 85)

In [143]:
class_ids = []
confidences = []
boxes = []
for out in outputs:
    for detection in out:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]
        if confidence > 0.53:
            center_x = int(detection[0] * width)
            center_y = int(detection[1] * height)
            w = int(detection[2] * width)
            h = int(detection[3] * height)

            x = int(center_x - w / 2)
            y = int(center_y - h / 2)
            boxes.append([x, y, w, h])

            confidences.append(float(confidence))
            class_ids.append(class_id)

In [144]:
colors = np.random.uniform(0, 255, size=(len(classes), 3))
indexes = cv.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

for i in range(len(boxes)):
    if i in indexes:
        x, y, w, h = boxes[i]
        label = str(classes[class_ids[i]])
        color = colors[class_ids[i]]
        cv.rectangle(image, (x, y), (x + w, y + h), color, 1)
        cv.putText(image, label, (x + 0, y + 0), cv.FONT_HERSHEY_PLAIN, .5, color, 1)
        # cv.putText(image, "{:.0f}".format(confidences[class_ids[i]] * 100), (x + 10, y + 60), cv.FONT_HERSHEY_PLAIN, .5, color, 1)

In [145]:
cv.imshow("Image", cv.resize(image, (1000, 750)))
cv.waitKey(0)
cv.destroyAllWindows()

## Detect on video

In [146]:
# cap = cv.VideoCapture("./opencv-notebook/videos/driving_camera.mp4")
cap = cv.VideoCapture(0)

In [147]:
while cap.isOpened():
    ret, frame = cap.read()
    if ret:
        img = cv.resize(frame, None, fx=0.8, fy=0.5)
        # img = frame

        height, width, channels = img.shape
        blob = cv.dnn.blobFromImage(img, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
        yolo_net.setInput(blob)
        outs = yolo_net.forward(output_layers)
        class_ids = []
        confidences = []
        boxes = []
        for out in outs:
            for detection in out:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                if confidence > 0.5:
                    center_x = int(detection[0] * width)
                    center_y = int(detection[1] * height)
                    w = int(detection[2] * width)
                    h = int(detection[3] * height)

                    x = int(center_x - w / 2)
                    y = int(center_y - h / 2)

                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)

        indexes = cv.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)
        font = cv.FONT_HERSHEY_PLAIN
        for i in range(len(boxes)):
            if i in indexes:
                x, y, w, h = boxes[i]
                label = str(classes[class_ids[i]])
                color = colors[class_ids[i]]
                cv.rectangle(img, (x, y), (x + w, y + h), color, 2)
                cv.putText(img, label, (x, y + 30), font, 1, color, 2)

        cv.imshow("Image", img)
        if cv.waitKey(1) == 27:
            break


cv.destroyAllWindows()
cap.release()